# Verification Collapse — 3-Iteration Sanity Check

This notebook runs a **3-iteration proof-of-concept** of the full experiment loop:
- Loads 50 GSM8K problems (instead of 500)
- Runs 3 iterations of generate → score → mine hard negatives → fine-tune
- Logs everything to W&B
- Checks that metrics are sensible before committing to the 20-iteration run


In [ ]:
import sys, os
# Make sure the project root is on the path
sys.path.insert(0, os.path.abspath(".."))

import yaml
from pathlib import Path
from copy import deepcopy

## 1. Load & override config for a quick run

In [ ]:
with open("../config.yaml") as f:
    cfg = yaml.safe_load(f)

# Override for the sanity check
cfg["data"]["subset_size"]   = 50     # smaller dataset
cfg["experiment"]["num_iterations"] = 3
cfg["experiment"]["checkpoint_interval"] = 3
cfg["experiment"]["hard_negative_injection_interval"] = 2  # trigger at iter 2
cfg["wandb"]["tags"] = cfg["wandb"]["tags"] + ["sanity-check"]

print(yaml.dump(cfg, default_flow_style=False))

## 2. Load GSM8K subset

In [ ]:
from src.gsm8k_loader import GSM8KDataset, answer_extractor, verify

dcfg = cfg["data"]
full_ds = GSM8KDataset(
    subset_size=dcfg["subset_size"],
    split="train",
    seed=dcfg["seed"],
    dataset_name=dcfg["dataset_name"],
    dataset_config=dcfg["dataset_config"],
    cache_path="../data/gsm8k_sanity.json",
)
train_ds, val_ds = full_ds.train_val_split(dcfg["train_ratio"])
print(f"Train: {len(train_ds.data)}  Val: {len(val_ds.data)}")
print("Sample:", train_ds[0]["question"][:120], "...")
print("Answer:", train_ds[0]["answer"])

### Quick answer-extractor smoke test

In [ ]:
test_cases = [
    (r"The answer is \boxed{42}.",         "42",  True),
    (r"So we get \boxed{1,234}.",          "1234", True),
    ("The total is 99 dollars.",           "99",  True),
    ("There are 3 apples and 5 oranges.",  "5",   True),
    ("No numbers here.",                   "42",  False),
]
for text, ref, expected in test_cases:
    extracted = answer_extractor(text)
    result    = verify(extracted, ref)
    status    = "OK" if result == expected else "FAIL"
    print(f"[{status}] extracted={extracted!r}  verify={result}  expected={expected}")

## 3. Load the model

In [ ]:
from src.verifier import ModelVerifier

verifier = ModelVerifier(cfg)
print("Model loaded.")

### Single-sample generation test

In [ ]:
sample = train_ds[0]
completion = verifier.generate([sample["prompt"]], max_new_tokens=128)[0]
print("PROMPT:")
print(sample["prompt"])
print("\nCOMPLETION:")
print(completion)
print("\nExtracted:", answer_extractor(completion))
print("Reference:", sample["answer"])
print("Correct:  ", verify(answer_extractor(completion), sample["answer"]))

## 4. Run 3-iteration experiment

In [ ]:
from src.experiment import VerificationCollapseExperiment

exp = VerificationCollapseExperiment(
    config=cfg,
    verifier=verifier,
    train_data=train_ds.data,
    val_data=val_ds.data,
)
exp.run(num_iterations=3)

## 5. Post-run checks

In [ ]:
# Verify checkpoint was saved
ckpt_dir = Path("../models")
checkpoints = sorted(ckpt_dir.glob("iter_*"))
print("Saved checkpoints:", checkpoints)

# Verify hard-negative bank grew
print("Hard-negative bank size:", len(exp.hard_negative_bank))

In [ ]:
# Quick metrics printout using utils directly
from src.utils import compute_gap, compute_self_score, compute_external_score

sample_scores = [1.0, 0.0, 1.0, 1.0, 0.0]  # mock
sample_completions = [
    r"Step 1: ... \boxed{42}",
    "The answer is 99.",
    r"\boxed{7}",
    r"\boxed{100}",
    "unknown",
]
sample_refs = ["42", "42", "7", "100", "5"]

ss = compute_self_score(sample_scores)
es = compute_external_score(sample_completions, sample_refs)
gap = compute_gap(ss, es)
print(f"self_score={ss:.2f}  external={es:.2f}  gap={gap:.2f}")